# U02.LAB07 · Visión computacional con Fashion-MNIST

**Asignatura:** INF-8239 Ciencia de Datos II  
**Unidad:** 02 · Procesamiento de Lenguaje Natural, redes y visión  
**Ejercicio:** U02.E04 · Clasificador visual con CNN y Model Card  
**Estudiante:** Manicaotex Mueses  
**Código:** AG-2554  

## Objetivo
Documentar de forma ejecutable las evidencias del LAB07: carga y auditoría de Fashion-MNIST, revisión de métricas, comparación entre baseline denso y CNN, análisis de errores visuales y validación del modelo guardado.


## 1. Preparar rutas e importar dependencias

El notebook detecta automáticamente la raíz del proyecto para funcionar tanto si Jupyter se abre desde la carpeta del proyecto como desde `notebooks/`.


In [1]:
from pathlib import Path
import json
import sys
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display, Image as IPImage

cwd = Path.cwd()

if (cwd / "reports").exists() and (cwd / "models").exists():
    ROOT = cwd
elif (cwd.parent / "reports").exists() and (cwd.parent / "models").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("No se encontró la raíz del proyecto. Abra el notebook desde INF8239_U02_CV o desde notebooks/.")

print("ROOT:", ROOT)
print("Python:", sys.version.split()[0])
print("Sistema:", platform.platform())
print("TensorFlow:", tf.__version__)
print("GPU detectada:", tf.config.list_physical_devices("GPU"))


ModuleNotFoundError: No module named 'tensorflow'

## 2. Cargar y auditar Fashion-MNIST

Fashion-MNIST contiene 60,000 imágenes de entrenamiento y 10,000 de prueba, en escala de grises y tamaño 28×28.


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test :", x_test.shape)
print("y_test :", y_test.shape)
print("Rango entrenamiento:", int(x_train.min()), "a", int(x_train.max()))
print("Número de clases:", len(np.unique(y_train)))

train_counts = pd.Series(y_train).value_counts().sort_index()
test_counts = pd.Series(y_test).value_counts().sort_index()

audit_df = pd.DataFrame({
    "clase_id": range(10),
    "clase": class_names,
    "train": train_counts.values,
    "test": test_counts.values
})
display(audit_df)


## 3. Muestra visual del dataset

In [ ]:
fig = plt.figure(figsize=(10, 6))
for i in range(12):
    ax = fig.add_subplot(3, 4, i + 1)
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(class_names[int(y_train[i])])
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Revisar las métricas generadas por el entrenamiento

Las métricas se leen directamente desde `reports/cv_metrics.json`, generado por `scripts/train_cv.py --epochs 8`.


In [ ]:
metrics_path = ROOT / "reports" / "cv_metrics.json"
with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

display(pd.DataFrame(metrics).T)


## 5. Comparación baseline denso vs CNN

In [ ]:
comparison = pd.DataFrame({
    "Modelo": ["Dense", "CNN"],
    "F1 macro": [metrics["dense"]["f1_macro"], metrics["cnn"]["f1_macro"]],
    "Parámetros": [metrics["dense"]["parameters"], metrics["cnn"]["parameters"]],
    "Entrenamiento (s)": [metrics["dense"]["train_seconds"], metrics["cnn"]["train_seconds"]],
    "Inferencia (ms/imagen)": [
        metrics["dense"]["inference_ms_per_image"],
        metrics["cnn"]["inference_ms_per_image"]
    ]
})

display(comparison)


In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(comparison["Modelo"], comparison["F1 macro"])
plt.ylabel("F1 macro")
plt.title("Comparación de desempeño")
plt.ylim(0, 1)
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(comparison["Modelo"], comparison["Entrenamiento (s)"])
plt.ylabel("Segundos")
plt.title("Costo de entrenamiento")
plt.show()


### Interpretación Green AI

En la ejecución registrada, el modelo denso obtuvo mayor F1 macro y menor tiempo de entrenamiento e inferencia. Aunque la CNN utilizó menos parámetros, su mayor costo temporal no estuvo acompañado por una mejora predictiva.


## 6. Matriz de confusión de la CNN

In [ ]:
confusion_path = ROOT / "reports" / "confusion_cnn.png"
display(IPImage(filename=str(confusion_path)))


## 7. Errores visuales de la CNN

In [ ]:
errors_path = ROOT / "reports" / "cnn_errors.png"
display(IPImage(filename=str(errors_path)))


### Análisis de errores

Se observan confusiones entre categorías visualmente semejantes, por ejemplo:

- `Coat` y `Pullover`
- `Shirt` con `Coat` o `T-shirt/top`
- `Dress` y `T-shirt/top`
- `Ankle boot`, `Sneaker` y `Sandal`

La clase 6 (`Shirt`) fue la más difícil para la CNN en la ejecución registrada, con recall aproximado de 0.371 y F1 aproximado de 0.425. Una métrica global puede ocultar debilidades importantes por clase.


## 8. Cargar el modelo CNN guardado y validar su salida

In [ ]:
model_path = ROOT / "models" / "best_cnn.keras"
model = tf.keras.models.load_model(model_path)

print("Modelo cargado:", model_path)
print("Forma de entrada :", model.input_shape)
print("Forma de salida  :", model.output_shape)
print("Parámetros       :", model.count_params())


In [ ]:
# Preparar un pequeño lote de prueba.
x_sample = x_test[:32].astype("float32") / 255.0

# Añadir canal si el modelo lo espera.
if len(model.input_shape) == 4 and x_sample.ndim == 3:
    x_sample = np.expand_dims(x_sample, axis=-1)

probs = model.predict(x_sample, verbose=0)

print("Forma de probabilidades:", probs.shape)
print("Valores finitos:", np.isfinite(probs).all())
print("Suma primera fila:", float(probs[0].sum()))
print("Clase predicha primera imagen:", int(np.argmax(probs[0])))
print("Clase real primera imagen:", int(y_test[0]))


## 9. Validar las pruebas automatizadas del proyecto

In [ ]:
import subprocess

result = subprocess.run(
    ["uv", "run", "pytest", "-q"],
    cwd=str(ROOT),
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)


## 10. Conclusión

El LAB07 permitió comparar de manera reproducible un baseline denso y una CNN sobre Fashion-MNIST. En la ejecución registrada, el baseline denso obtuvo un F1 macro de aproximadamente **0.8634**, mientras que la CNN obtuvo aproximadamente **0.7889**. El modelo denso también presentó menor tiempo de entrenamiento e inferencia.

La CNN utilizó menos parámetros, pero ese ahorro no compensó el mayor costo temporal ni el menor desempeño predictivo observado. El análisis de errores mostró dificultades especialmente en categorías visualmente parecidas, con la clase `Shirt` como la más débil.

Estos resultados deben interpretarse únicamente dentro de Fashion-MNIST. Antes de utilizar un modelo semejante en otro dominio sería necesario validar datos representativos, revisar métricas por clase, estudiar los errores y justificar el costo computacional.


## 11. Uso de IA

Se utilizó Claude y ChatGPT como apoyo para interpretar instrucciones, explicar resultados, revisar métricas, analizar errores visuales y organizar la documentación. Las métricas y evidencias reportadas fueron verificadas mediante la ejecución local del proyecto.
